In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/her-will-ai-for-digital-safety-datathon-2026/sample_submission.csv
/kaggle/input/competitions/her-will-ai-for-digital-safety-datathon-2026/train.csv
/kaggle/input/competitions/her-will-ai-for-digital-safety-datathon-2026/test.csv


In [2]:
pip install torch torchvision torchaudio transformers pandas scikit-learn thop tqdm

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from thop import profile
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

# 1. Configuration & Hyperparameters

In [4]:

MODEL_NAME = "distilbert-base-multilingual-cased" # Lightweight multilingual model
MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 3
LEARNING_RATE = 2e-5
NUM_CLASSES = 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

Using device: cpu


# 2. Data Preparation


In [5]:
# Load datasets
train_df = pd.read_csv('/kaggle/input/competitions/her-will-ai-for-digital-safety-datathon-2026/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/her-will-ai-for-digital-safety-datathon-2026/test.csv')

# Drop nulls just in case
train_df = train_df.dropna(subset=['text', 'y'])
test_df = test_df.dropna(subset=['text'])

# Stratified split to maintain class distribution in validation
train_data, val_data = train_test_split(
    train_df, test_size=0.15, stratify=train_df['y'], random_state=42
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class SocialMediaDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=128):
        self.texts = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True) if labels is not None else None
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        item = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }
        
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
            
        return item

# Create DataLoaders
train_dataset = SocialMediaDataset(train_data['text'], train_data['y'], tokenizer, MAX_LEN)
val_dataset = SocialMediaDataset(val_data['text'], val_data['y'], tokenizer, MAX_LEN)
test_dataset = SocialMediaDataset(test_df['text'], labels=None, tokenizer=tokenizer, max_len=MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# 3. Model Initialization & NOP Calculation

In [6]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_CLASSES)

# Rule Requirement: Calculate NOP using thop
def measure_nop(model, inputs):
    # thop expects a tuple of inputs for the model's forward pass
    flops, params = profile(model, inputs=(inputs['input_ids'], inputs['attention_mask']), verbose=False)
    return flops, params

# Create dummy input for NOP calculation
dummy_inputs = {
    'input_ids': torch.randint(0, tokenizer.vocab_size, (1, MAX_LEN)),
    'attention_mask': torch.ones(1, MAX_LEN)
}

flops, total_params = measure_nop(model, dummy_inputs)
print(f"Model Parameters (NOP): {total_params}")

model = model.to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

model.safetensors:   0%|          | 0.00/542M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model Parameters (NOP): 43121667.0


# 4. Training Loop

In [7]:

print("Starting Training...")
for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0
    
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_train_loss += loss.item()
        
        loss.backward()
        optimizer.step()
        
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Validation step
    model.eval()
    val_preds, val_true = [], []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            
            val_preds.extend(preds)
            val_true.extend(labels.cpu().numpy())
            
    val_f1 = f1_score(val_true, val_preds, average='macro')
    print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Macro F1: {val_f1:.4f}")

Starting Training...


Epoch 1/3 [Val]: 100%|██████████| 225/225 [07:59<00:00,  2.13s/it]


Epoch 1 | Train Loss: 0.8805 | Val Macro F1: 0.4782


Epoch 2/3 [Val]: 100%|██████████| 225/225 [07:52<00:00,  2.10s/it]


Epoch 2 | Train Loss: 0.7803 | Val Macro F1: 0.5321


Epoch 3/3 [Val]: 100%|██████████| 225/225 [07:51<00:00,  2.10s/it]

Epoch 3 | Train Loss: 0.6654 | Val Macro F1: 0.5153


# 5. Inference & Submission

In [8]:

print("Generating predictions for test set...")
model.eval()
test_preds = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        test_preds.extend(preds)

# Create submission DataFrame
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'y_pred': test_preds,
    'parameters': [total_params] * len(test_preds) # Repeat NOP for each row as required
})

Generating predictions for test set...


Testing: 100%|██████████| 374/374 [13:10<00:00,  2.11s/it]


In [9]:
# Save to CSV
submission_file = 'submission.csv'
submission_df.to_csv(submission_file, index=False)
print(f"Submission saved to {submission_file}. Ready for upload!")

Submission saved to submission.csv. Ready for upload!
